In [14]:
from mask_utils import merge_masks
from point_utils import get_da3_pointclouds
from data import *
import os
import cv2
import numpy as np
from tracker import Simple3DTracker
from track_vis_utils import *

In [2]:
file_dir_name = '2'
frame_path = '../data/%s/' %(file_dir_name)
frame_names = sorted(os.listdir(frame_path))

In [3]:
frame_names = sorted(os.listdir(frame_path))
images = []
image_paths = []
#getting images
for name in frame_names:
    images.append(cv2.imread('%s/%s' %(frame_path,name)))
    image_paths.append('%s/%s' %(frame_path,name))

In [4]:
#getting tracks
tracks_dir = 'sam_tracks/%s/' %(file_dir_name)
track_files = sorted(os.listdir(tracks_dir))
raw_tracks = {}
for i in range(len(track_files)):
    track = np.load('%s/%d.npz' %(tracks_dir,i),allow_pickle=True)['arr_0']
    if track.tolist()['cls']!=0:
        raw_tracks[i]= track.tolist()

In [5]:
max(raw_tracks.keys())

40

In [6]:
tracks = merge_masks(raw_tracks)

In [7]:
#getting frame embeddings
emb_dir = 'sam_embs/%s/' %(file_dir_name)
frame_embeds = []
emb_files = sorted(os.listdir(emb_dir))
for i in range(len(emb_files)):
    frame_embeds.append(np.load('%s/%d.npz' %(emb_dir,i),allow_pickle=True)['arr_0'])

In [8]:
da_res_dir = 'da3_outputs_new/%s/' %(file_dir_name)
depth_dir = da_res_dir+'/results_output/'
extrinsics_file = da_res_dir+'camera_poses.txt'

In [9]:
#pointclouds
points_per_frame, points_per_frame_masks,h,w = get_da3_pointclouds(depth_dir,extrinsics_file,len(frame_names))

In [10]:
#text_embeddings
text_embs = get_text_embeddings(tracks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
tracker=Simple3DTracker()

In [12]:
track_voxels_history = {}

for i in range(len(images)):
    detections = []
    print(i)
    for j in tracks.keys():
        if i in tracks[j]['masks'].keys():
            det ={}
            det['sam_id']=j
            det['cls']=tracks[j]['cls']
            det['mask']=tracks[j]['masks'][i]
            pcd0 =get_obj_point_cloud(images[i],points_per_frame[i],
                                              points_per_frame_masks[i],
                                              det['mask'],h,w)
            det['points']=np.asarray(pcd0.points)
            if len(det['points'])<5:
                continue
            det['embedding']=get_object_embedding(frame_embeds[i],det['mask']).reshape(1,-1)
            det['text_embedding']=text_embs[det['cls']]
            detections.append(det)
    tracker.update(detections,i)

    # Сохраняем voxelmap каждого существующего трека на каждом кадре
    existing_tracks = {int(t.id): t for t in tracker.tracks}
    for t in tracker.lost_tracks:
        existing_tracks[int(t.id)] = t

    for tid, t in existing_tracks.items():
        pcd_global = t.voxels.get_pcd()
        pcd_global = np.asarray(pcd_global.points)
        if len(pcd_global) == 0:
            continue
        pts_global = pcd_global.astype(np.float32)
        if tid not in track_voxels_history:
            track_voxels_history[tid] = {}
        track_voxels_history[tid][int(i)] = pts_global


0
1
carpet 0
2
carpet 0
3
carpet 0
4
carpet 0
5
carpet 0
6
carpet 0
7
carpet 0
8
carpet 0
9
carpet 0
10
carpet 0
door 1
11
carpet 0
door 1
12
carpet 0
carpet 2
door 1
door 3
13
carpet 0
carpet 2
door 1
door 3
14
carpet 0
carpet 2
door 1
door 3
15
carpet 0
carpet 2
door 1
door 3
16
carpet 0
carpet 2
door 1
door 3
17
carpet 2
door 1
door 3
18
carpet 2
door 1
door 3
19
door 1
carpet 2
door 3
20
door 1
carpet 2
door 3
21
carpet 2
door 1
door 3
22
door 1
door 3
23
door 1
door 3
24
door 1
door 3
25
door 1
door 3
26
door 1
door 3
27
door 1
door 3
28
door 1
door 3
29
door 1
door 3
30
door 1
door 3
31
door 1
door 3
door 7
32
door 1
door 3
door 7
re-init track 7
33
door 1
door 3
door 9
34
door 1
door 3
door 9
35
carpet 2
door 1
door 3
bed 4
36
carpet 2
door 1
door 3
bed 4
window 8
37
carpet 2
door 1
door 3
bed 4
38
carpet 2
door 1
door 3
bed 4
39
carpet 2
door 1
door 3
bed 4
40
carpet 2
door 1
door 3
bed 4
41
carpet 2
door 1
door 3
bed 4
42
carpet 2
door 1
bed 4
43
carpet 2
door 1
bed 4
44
carpe

In [15]:
all_tracks = get_current_tracks(tracker)

In [18]:
masked_images = visualize_tracks(images,all_tracks,'mask')

In [19]:
from pathlib import Path
out_dir = Path('vis_data/outputs/2')
track_out_dir = Path('vis_data/track_outputs/2')
out_meta_dir = Path('vis_data/meta_outputs/2')
out_filtered_dir = Path('vis_data/filtered_outputs/2')
out_point_dir = Path('vis_data/point_outputs/2')
out_meta_dir.mkdir(parents=True, exist_ok=True)
out_point_dir.mkdir(parents=True, exist_ok=True)
out_filtered_dir.mkdir(parents=True, exist_ok=True)
out_dir.mkdir(parents=True, exist_ok=True)

In [20]:
for i in range(len(masked_images)):
    cv2.imwrite(out_dir / f"{i}.png", masked_images[i])

In [21]:
import shutil
shutil.rmtree(track_out_dir, ignore_errors=True)
track_out_dir.mkdir(parents=True, exist_ok=True)
for i in tracks.keys():
    np.savez(track_out_dir / f"{i}.npz", tracks[i],pickle=True)

In [22]:
import json

track_names = {int(idx):{'cls':t['cls']} for idx, t in tracks.items()}
with open(out_meta_dir / 'track_names.json', 'w') as f:
    json.dump(track_names, f, indent=4)


In [23]:
for i, mask in enumerate(points_per_frame_masks):
    np.save(out_filtered_dir / f"{i}_filtered.npy", mask)

In [24]:
for i, points in enumerate(points_per_frame):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    o3d.io.write_point_cloud(str(out_point_dir / f"{i}.ply"), pcd)

In [25]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def show_pointcloud_in_notebook(pcd, max_points=100000, downsample_step=None):
    """Визуализация одного Open3D PointCloud в Jupyter через Plotly (как в lab/notes/da3.ipynb)."""
    pts = np.asarray(pcd.points)
    clr = np.asarray(pcd.colors) if pcd.has_colors() else np.ones((len(pts), 3)) * 0.7
    n = len(pts)
    if downsample_step is None:
        downsample_step = max(1, n // max_points)
    step = downsample_step
    pts, clr = pts[::step], clr[::step]
    rgb_str = np.array([f"rgb({r*255:.0f},{g*255:.0f},{b*255:.0f})" for r, g, b in clr])
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=pts[:, 0],
                y=pts[:, 1],
                z=pts[:, 2],
                mode="markers",
                marker=dict(size=1.5, color=rgb_str),
            )
        ]
    )
    fig.update_layout(
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            bgcolor="rgb(20,20,20)",
        ),
        margin=dict(l=0, r=0, t=0, b=0),
        height=500,
    )
    return fig


def _subsample_points(pts, max_n):
    if len(pts) <= max_n:
        return pts
    idx = np.random.choice(len(pts), max_n, replace=False)
    return pts[idx]


def track_point_clouds_at_frame(
    frame_idx, all_tracks, points_per_frame, points_per_frame_masks, h, w
):
    """3D-точки по каждому треку на кадре frame_idx (тот же pipeline, что при tracker.update)."""
    clouds = []
    for track in all_tracks:
        if frame_idx not in track.masks:
            continue
        pts = get_obj_point_cloud(
            points_per_frame[frame_idx],
            points_per_frame_masks[frame_idx],
            track.masks[frame_idx],
            h,
            w,
        )
        if len(pts) == 0:
            continue
        clouds.append((track.id, pts))
    return clouds


def tracks_to_merged_open3d_pcd(
    frame_idx,
    all_tracks,
    points_per_frame,
    points_per_frame_masks,
    h,
    w,
    track_colors=None,
):
    """Одно o3d.geometry.PointCloud: все треки на кадре, цвет закреплён за id трека."""
    if track_colors is None:
        track_colors = distinctipy.get_colors(len(all_tracks))
    id_to_j = {t.id: j for j, t in enumerate(all_tracks)}
    clouds = track_point_clouds_at_frame(
        frame_idx, all_tracks, points_per_frame, points_per_frame_masks, h, w
    )
    merged = o3d.geometry.PointCloud()
    if not clouds:
        return merged
    parts = []
    cols = []
    for tid, pts in clouds:
        j = id_to_j.get(tid, 0)
        c = np.asarray(track_colors[j], dtype=np.float64)
        parts.append(pts)
        cols.append(np.tile(c, (len(pts), 1)))
    all_pts = np.vstack(parts)
    all_cols = np.vstack(cols)
    merged.points = o3d.utility.Vector3dVector(all_pts)
    merged.colors = o3d.utility.Vector3dVector(all_cols)
    return merged


def show_tracks_pointclouds_at_frame(
    frame_idx,
    all_tracks,
    points_per_frame,
    points_per_frame_masks,
    h,
    w,
    track_colors=None,
    max_points_per_track=20000,
    title=None,
):
    """Один момент времени: треки разными цветами (Plotly)."""
    if track_colors is None:
        track_colors = distinctipy.get_colors(len(all_tracks))
    id_to_j = {t.id: j for j, t in enumerate(all_tracks)}
    fig = go.Figure()
    clouds = track_point_clouds_at_frame(
        frame_idx, all_tracks, points_per_frame, points_per_frame_masks, h, w
    )
    for tid, pts in clouds:
        j = id_to_j.get(tid, 0)
        c = track_colors[j]
        pts_ds = _subsample_points(pts, max_points_per_track)
        rgb_str = f"rgb({c[0]*255:.0f},{c[1]*255:.0f},{c[2]*255:.0f})"
        fig.add_trace(
            go.Scatter3d(
                x=pts_ds[:, 0],
                y=pts_ds[:, 1],
                z=pts_ds[:, 2],
                mode="markers",
                marker=dict(size=2, color=rgb_str),
                name=f"track {tid}",
            )
        )
    fig.update_layout(
        title=title if title is not None else f"Кадр {frame_idx}",
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            bgcolor="rgb(20,20,20)",
        ),
        margin=dict(l=0, r=0, t=50, b=0),
        height=560,
    )
    return fig


def show_tracks_pointclouds_at_frames(
    frame_indices,
    all_tracks,
    points_per_frame,
    points_per_frame_masks,
    h,
    w,
    track_colors=None,
    max_points_per_track=15000,
):
    """Несколько моментов времени: подряд в subplot (одинаковые цвета для одного track id)."""
    if track_colors is None:
        track_colors = distinctipy.get_colors(len(all_tracks))
    cols = len(frame_indices)
    fig = make_subplots(
        rows=1,
        cols=cols,
        specs=[[{"type": "scatter3d"}] * cols],
        subplot_titles=[f"Кадр {fi}" for fi in frame_indices],
        horizontal_spacing=0.03,
    )
    id_to_j = {t.id: j for j, t in enumerate(all_tracks)}
    for col, frame_idx in enumerate(frame_indices, start=1):
        clouds = track_point_clouds_at_frame(
            frame_idx, all_tracks, points_per_frame, points_per_frame_masks, h, w
        )
        for tid, pts in clouds:
            j = id_to_j.get(tid, 0)
            c = track_colors[j]
            pts_ds = _subsample_points(pts, max_points_per_track)
            rgb_str = f"rgb({c[0]*255:.0f},{c[1]*255:.0f},{c[2]*255:.0f})"
            fig.add_trace(
                go.Scatter3d(
                    x=pts_ds[:, 0],
                    y=pts_ds[:, 1],
                    z=pts_ds[:, 2],
                    mode="markers",
                    marker=dict(size=2, color=rgb_str),
                    name=f"track {tid}",
                    legendgroup=str(tid),
                    showlegend=(col == 1),
                ),
                row=1,
                col=col,
            )
    fig.update_layout(height=520, margin=dict(l=0, r=0, t=50, b=0))
    fig.update_scenes(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor="rgb(20,20,20)",
    )
    return fig


# Пример (после all_tracks, points_per_frame, h, w):
# mid = len(images) // 2
# fig_3d = show_tracks_pointclouds_at_frames(
#     [0, mid, len(images) - 1], all_tracks, points_per_frame, points_per_frame_masks, h, w
# )
# fig_3d.show()
# pcd_tracks = tracks_to_merged_open3d_pcd(mid, all_tracks, points_per_frame, points_per_frame_masks, h, w)
# o3d.visualization.draw_geometries([pcd_tracks])


In [28]:
extrinsics = []

# pose extraction
with open(extrinsics_file, "r") as f:
    all_poses = f.read().split("\n")
for i in range(len(images)):
    pose = all_poses[i]
    pose_np = np.array(pose.split(" ")).astype(float).reshape((4, 4))
    extrinsics.append(pose_np)

In [30]:
res_dir = da_res_dir

In [31]:
import json
import pickle
from pathlib import Path

import numpy as np

# Данные для lab/scripts/tracker_layers_rerun.py (камера, сцена, треки, combined ply)
RERUN_EXPORT_DIR = out_point_dir / "rerun_export"
RERUN_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

tracks_serial = []
for t in all_tracks:
    tid = int(t.id)
    voxels_by_frame = {
        int(frame_idx): np.asarray(pts, dtype=np.float32)
        for frame_idx, pts in track_voxels_history.get(tid, {}).items()
    }
    tracks_serial.append(
        {
            "id": tid,
            "masks": {int(k): v for k, v in t.masks.items()},
            "voxels_by_frame": voxels_by_frame,
        }
    )
with open(RERUN_EXPORT_DIR / "tracks.pkl", "wb") as f:
    pickle.dump(tracks_serial, f, protocol=pickle.HIGHEST_PROTOCOL)

np.save(RERUN_EXPORT_DIR / "extrinsics.npy", np.stack(extrinsics, axis=0))

with open(RERUN_EXPORT_DIR / "points_per_frame.pkl", "wb") as f:
    pickle.dump(points_per_frame, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(RERUN_EXPORT_DIR / "points_per_frame_masks.pkl", "wb") as f:
    pickle.dump(points_per_frame_masks, f, protocol=pickle.HIGHEST_PROTOCOL)

track_colors = distinctipy.get_colors(len(all_tracks))
np.save(RERUN_EXPORT_DIR / "track_colors.npy", np.array(track_colors))

config = {
    "depth_dir": depth_dir,
    "n_frames": len(images),
    "fps": 5.0,
    "scene_sub": 2,
    "track_sub": 1,
    "track_voxels_sub": 1,
    "track_voxels_radius": 0.004,
    "combined_sub": 2,
    "combined_alpha": 0.35,
    "combined_pcd_paths": [
        str(Path(res_dir) / "pcd" / "combined_pcd.ply"),
        str(Path("/home/kondrashov_k/mipt/lab/pcd/combined_pcd.ply")),
    ],
}
with open(RERUN_EXPORT_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("Saved rerun export to", RERUN_EXPORT_DIR)


Saved rerun export to vis_data/point_outputs/2/rerun_export


In [32]:
import subprocess
import sys
from pathlib import Path

# После ячейки сохранения export; нужны: rerun-sdk, те же пути к depth_dir на диске
SCRIPT = Path("scripts/tracker_layers_rerun.py")
RERUN_EXPORT_DIR = out_point_dir / "rerun_export"
TRACK_RRD_OUT = out_point_dir / "tracker_layers.rrd"
RUNPY_RRD_OUT = out_point_dir / "runpy_layers.rrd"
cmd = [
    sys.executable,
    str(SCRIPT),
    "--export-dir",
    str(RERUN_EXPORT_DIR),
    "--save",
    str(TRACK_RRD_OUT)
]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)
print("Wrote", TRACK_RRD_OUT)


Running: /home/zimina_sv/miniconda3/envs/sam3/bin/python3 scripts/tracker_layers_rerun.py --export-dir vis_data/point_outputs/2/rerun_export --save vis_data/point_outputs/2/tracker_layers.rrd
Logged combined PCD: da3_outputs_new/2/pcd/combined_pcd.ply (1226421 pts)
Saved: vis_data/point_outputs/2/tracker_layers.rrd
Wrote vis_data/point_outputs/2/tracker_layers.rrd


In [ ]:
# Экспорт в формат, который потребляет адаптер runpy_rerun_adapter.py
# (далее адаптер прокидывает данные в yolo_sgg/rerun_utils.RerunVisualizer без изменения run.py)
import json
import pickle
from pathlib import Path

import numpy as np

RUNPY_EXPORT_DIR = out_point_dir / "runpy_export"
RUNPY_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Берем intrinsics для RerunVisualizer из первого depth npz
intr0 = np.load(depth_dir + "frame_0.npz")["intrinsics"].astype(np.float32)
fx = float(intr0[0, 0])
fy = float(intr0[1, 1])
cx = float(intr0[0, 2])
cy = float(intr0[1, 2])
img_h, img_w = images[0].shape[:2]


def _cls_at_frame(track, frame_idx: int):
    if frame_idx in track.cls:
        return track.cls[frame_idx]
    prev = [k for k in track.cls.keys() if k <= frame_idx]
    if not prev:
        return None
    return track.cls[max(prev)]


n_frames = len(images)
objects_by_frame = []
masks_clean_by_frame = []
track_ids_by_frame = []
class_names_by_frame = []

for i in range(n_frames):
    frame_objs = []
    frame_masks = []
    frame_track_ids = []
    frame_class_names = []

    for t in all_tracks:
        tid = int(t.id)
        pts = track_voxels_history.get(tid, {}).get(int(i))
        if pts is None:
            continue

        pts = np.asarray(pts, dtype=np.float32)
        if len(pts) == 0:
            continue

        pmin = pts.min(axis=0).astype(np.float32)
        pmax = pts.max(axis=0).astype(np.float32)

        cls_val = _cls_at_frame(t, i)
        cls_name = None if cls_val is None else str(cls_val)

        frame_objs.append(
            {
                "global_id": tid,
                "points": pts,
                "class_name": cls_name,
                "bbox_3d": {
                    "aabb": {
                        "min": pmin.tolist(),
                        "max": pmax.tolist(),
                    }
                },
                "visible_current_frame": True,
                "observation_count": int(sum(1 for k in t.masks.keys() if k <= i)),
            }
        )

        if i in t.masks:
            frame_masks.append(t.masks[i])
            frame_track_ids.append(tid)
            frame_class_names.append(cls_name)

    objects_by_frame.append(frame_objs)
    masks_clean_by_frame.append(frame_masks)
    track_ids_by_frame.append(np.array(frame_track_ids, dtype=np.int32))
    class_names_by_frame.append(frame_class_names)

with open(RUNPY_EXPORT_DIR / "runpy_objects.pkl", "wb") as f:
    pickle.dump(objects_by_frame, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(RUNPY_EXPORT_DIR / "runpy_masks_clean.pkl", "wb") as f:
    pickle.dump(masks_clean_by_frame, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(RUNPY_EXPORT_DIR / "runpy_track_ids.pkl", "wb") as f:
    pickle.dump(track_ids_by_frame, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(RUNPY_EXPORT_DIR / "runpy_class_names.pkl", "wb") as f:
    pickle.dump(class_names_by_frame, f, protocol=pickle.HIGHEST_PROTOCOL)

np.save(RUNPY_EXPORT_DIR / "runpy_T_w_c.npy", np.stack(extrinsics, axis=0).astype(np.float32))

cfg = {
    "recording_id": "tracker_runpy_adapter",
    "img_w": int(img_w),
    "img_h": int(img_h),
    "fx": fx,
    "fy": fy,
    "cx": cx,
    "cy": cy,
    "rgb_paths": image_paths,
    "vis_edges": False,
}
with open(RUNPY_EXPORT_DIR / "runpy_export_config.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)

# Опционально: можно добавить runpy_graph_edges.json со списком ребер {src, dst, label}
print("Saved run.py-compatible export to", RUNPY_EXPORT_DIR)